In [ ]:
import re

In [ ]:
import json

In [ ]:
import tqdm

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
import pandas as pd

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

In [ ]:
df = pd.read_excel("data_raw.xlsx", header=0)

In [ ]:
df

In [ ]:
df = df.fillna("")

In [ ]:
df[df.duplicated(subset=["title_rus"])]["title_rus"].nunique()

In [ ]:
df.shape

In [ ]:
df = df.drop_duplicates(subset=["title_rus"], inplace=False).reset_index(
    drop=True, inplace=False
)

In [ ]:
df.shape

In [ ]:
df

In [ ]:
df[df.duplicated(subset=["title_rus"])]

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can validate whether or not a provided student project title is a valid project title in russian.
In case title is valid, return 1. Otherwise 0.
Return strictly JSON as in example below.
```json
{"is_valid": 1}
```
"""

In [ ]:
messages = []

In [ ]:
for row in df.iterrows():
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row[1]["title_rus"]},
        )
    )

In [ ]:
idx = []

In [ ]:
for message in tqdm.tqdm(messages):
    text = processor.apply_chat_template(
        message, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(**inputs, max_new_tokens=1024)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

    json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    if json_match:
        json_str = json_match.group(1)
    else:
        json_str = response

    try:
        answer = json.loads(json_str)
        idx.append(answer.get("is_valid", 0))
    except Exception:
        idx.append(0)

In [ ]:
idx = [True if x else False for x in idx]

In [ ]:
len(df)

In [ ]:
df = df[idx].reset_index(drop=True)

In [ ]:
len(df)

In [ ]:
df[["title_rus", "title_eng", "annotation", "description"]].to_excel(
    "data_clean.xlsx", index=False, header=True
)